In [43]:
import pandas as pd
df = pd.read_csv('../Dataset/Loan_default.csv')
df = df.drop(['LoanID'],axis = 1)

# ML flow setup

In [44]:
import mlflow
import os 

proj_root_path = os.path.abspath(os.path.join(os.getcwd(),".."))
tracking_uri = os.path.join(proj_root_path,"mlruns")

mlflow.set_tracking_uri(f"file:{tracking_uri}")
mlflow.set_experiment("my_experiment")


<Experiment: artifact_location='file:d:\\projects\\home_default\\mlruns/801978657764068180', creation_time=1769985019900, experiment_id='801978657764068180', last_update_time=1769985019900, lifecycle_stage='active', name='my_experiment', tags={'mlflow.experimentKind': 'custom_model_development'}>

# Missing values imputation/dropping
- Since no missing values no need to complete this

In [45]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 255347 entries, 0 to 255346
Data columns (total 17 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Age             255347 non-null  int64  
 1   Income          255347 non-null  int64  
 2   LoanAmount      255347 non-null  int64  
 3   CreditScore     255347 non-null  int64  
 4   MonthsEmployed  255347 non-null  int64  
 5   NumCreditLines  255347 non-null  int64  
 6   InterestRate    255347 non-null  float64
 7   LoanTerm        255347 non-null  int64  
 8   DTIRatio        255347 non-null  float64
 9   Education       255347 non-null  str    
 10  EmploymentType  255347 non-null  str    
 11  MaritalStatus   255347 non-null  str    
 12  HasMortgage     255347 non-null  str    
 13  HasDependents   255347 non-null  str    
 14  LoanPurpose     255347 non-null  str    
 15  HasCoSigner     255347 non-null  str    
 16  Default         255347 non-null  int64  
dtypes: float64(2), int64(

## Feature Classification
- Numeric (scaled): Large-range continuous financial variables
- Numeric (unscaled): Bounded or count-based variables
- Ordinal categorical: Ordered categories
- Nominal categorical: Unordered categories
- Binary categorical: Explicit 0/1 mapping


# train test split

In [46]:
from sklearn.model_selection import train_test_split
X = df.drop('Default',axis = 1)
y = df['Default']

X_train,X_test,y_train,y_test= train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify=y,
    shuffle = True
)

In [47]:
num_unscaled = [
    "Age",
    "MonthsEmployed",
    "NumCreditLines",
    "LoanTerm"
]

num_scaled = [
    "Income",
    "LoanAmount",
    "CreditScore",
    "InterestRate",
    "DTIRatio"
]

ordinal_cols = [
    "Education"
]
nominal_cols = [
    "EmploymentType",
    "MaritalStatus",
    "LoanPurpose"
]
binary_cols = [
    "HasMortgage",
    "HasDependents",
    "HasCoSigner"
]

- Numeric scaling is applied to num_scaled where numeric stability is important.
- Ordinal categories are encoded with ordering.
- Nominal categories are one-hot encoded without dropping reference categories.
- Binary features are mapped directly to 0/1 for clarity.


In [48]:
from sklearn.preprocessing import OrdinalEncoder

education_order = [
    ["High School", "Bachelor's", "Master's", "PhD"]
]

ordinal_encoder = OrdinalEncoder(categories=education_order)

binary_encoder = OrdinalEncoder(
    categories=[['No','Yes']] * 3 # No -> 0 and yes -> 1
)


In [49]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num_scaled", StandardScaler(), num_scaled),
        ("binary",binary_encoder, binary_cols),
        ("num_unscaled", "passthrough", num_unscaled),
        ("ordinal", ordinal_encoder, ordinal_cols),
        ("nominal", OneHotEncoder(handle_unknown="ignore", sparse_output=False,drop='first'), nominal_cols),
    ],
    remainder="drop",
    # verbose=True
)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num_scaled", "passthrough", num_scaled),
        ("num_unscaled", "passthrough", num_unscaled),
        ("binary",binary_encoder, binary_cols),
        ("ordinal", ordinal_encoder, ordinal_cols),
        ("nominal", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominal_cols),
    ],
    remainder="drop",
    # verbose=True
)

In [ ]:
import mlflow
import mlflow.sklearn

from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix
)
import pandas as pd
import numpy as np

with mlflow.start_run(run_name="Baseline"):
    pipe = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", LogisticRegression(
                max_iter=1000,
                solver="lbfgs",
                n_jobs=-1
            ))
        ]
)

    cv_scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=5,
        scoring="roc_auc"
    )
    pipe.fit(X_train,y_train)

    # Log model parameters
    mlflow.log_params(pipe.named_steps["model"].get_params())
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("cv_scoring", "roc_auc")
    # mlflow.log_param("random_state", 42)
    mlflow.log_param("preprocessor", "StandardScaler + OrdinalEncoder + OneHot(drop=first) + BinaryOrdinal")

    mlflow.log_metric("cv_roc_auc_mean", cv_scores.mean())
    mlflow.log_metric("cv_roc_auc_std", cv_scores.std())

    y_test_proba = pipe.predict_proba(X_test)[:, 1]

    test_roc_auc = roc_auc_score(y_test, y_test_proba)

    mlflow.log_metric("test_roc_auc", test_roc_auc)

    mlflow.sklearn.log_model(
        pipe,
        artifact_path="model"
    )

    y_test_pred = (y_test_proba >= 0.5).astype(int)

    mlflow.log_metric("test_precision", precision_score(y_test, y_test_pred))
    mlflow.log_metric("test_recall", recall_score(y_test, y_test_pred))
    mlflow.log_metric("test_f1", f1_score(y_test, y_test_pred))
    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_test_pred))

    # Confusion matrix as artifact
    cm = confusion_matrix(y_test, y_test_pred)
    cm_df = pd.DataFrame(cm, index=["Actual_0", "Actual_1"], columns=["Pred_0", "Pred_1"])
    mlflow.log_dict(cm_df.to_dict(), "confusion_matrix.json")

    # ROC curve and KS statistic
    fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
    ks = np.max(tpr - fpr)
    roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr, "thresholds": thresholds})
    roc_df.to_csv("roc_curve.csv", index=False)
    mlflow.log_artifact("roc_curve.csv")
    mlflow.log_dict(roc_df.to_dict(), "roc_curve.json")
    mlflow.log_metric("test_ks", float(ks))

    print("CV ROC-AUC scores:", cv_scores)
    print("Mean ROC-AUC:", cv_scores.mean())
    print("test_roc_auc", test_roc_auc)
    print("test_ks", ks)

d:\projects\home_default\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\projects\home_default\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\projects\home_default\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\projects\home_default\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please lea

CV ROC-AUC scores: [0.74840325 0.74810731 0.74031707 0.74850712 0.74656279]
Mean ROC-AUC: 0.7463795097529526
test_roc_auc 0.7530980990623658
test_ks 0.37763764449367954


In [51]:
print("test_roc_auc", test_roc_auc)

test_roc_auc 0.7530980990623658


# Day 5

In [52]:
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
import mlflow.xgboost

from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
import pandas as pd
import lightgbm as lgb
import xgboost as xgb


In [ ]:
with mlflow.start_run(run_name="Pipeline_LightGBM"):

    pipe_lgb = Pipeline(
        steps=[
            ("preprocess",preprocessor_tree ),
            ("model", lgb.LGBMClassifier(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=-1,
                num_leaves=31,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary",
                random_state=42
            ))
        ]
    )

    pipe_lgb.fit(X_train, y_train)

    y_train_proba = pipe_lgb.predict_proba(X_train)[:, 1]
    y_test_proba = pipe_lgb.predict_proba(X_test)[:, 1]

    train_auc = roc_auc_score(y_train, y_train_proba)
    test_auc = roc_auc_score(y_test, y_test_proba)

    # ---- MLflow logging ----
    mlflow.log_params(pipe_lgb.named_steps["model"].get_params())
    mlflow.log_metric("train_roc_auc", train_auc)
    mlflow.log_metric("test_roc_auc", test_auc)

    # Additional metrics
    y_test_pred = (y_test_proba >= 0.5).astype(int)
    mlflow.log_metric("test_precision", precision_score(y_test, y_test_pred))
    mlflow.log_metric("test_recall", recall_score(y_test, y_test_pred))
    mlflow.log_metric("test_f1", f1_score(y_test, y_test_pred))
    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_test_pred))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_test_pred)
    cm_df = pd.DataFrame(cm, index=["Actual_0", "Actual_1"], columns=["Pred_0", "Pred_1"])
    mlflow.log_dict(cm_df.to_dict(), "confusion_matrix_lgb.json")

    # KS statistic and ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
    ks = np.max(tpr - fpr)
    mlflow.log_metric("test_ks", float(ks))
    roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr, "thresholds": thresholds})
    roc_df.to_csv("roc_curve_lgb.csv", index=False)
    mlflow.log_dict(roc_df.to_dict(), "roc_curve_lgb.json")

    mlflow.sklearn.log_model(pipe_lgb, artifact_path="model")

    # ---- Feature importance (post-preprocessing) ----
    feature_names = pipe_lgb.named_steps["preprocess"].get_feature_names_out()

    fi = pd.DataFrame({
        "feature": feature_names,
        "importance": pipe_lgb.named_steps["model"].feature_importances_
    }).sort_values("importance", ascending=False)

    fi.to_csv("lgb_feature_importance.csv", index=False)
    mlflow.log_dict(fi.to_dict(), "lgb_feature_importance.json")

    print("LightGBM Train ROC-AUC:", train_auc)
    print("LightGBM Test ROC-AUC:", test_auc)
    print("LightGBM Test KS:", ks)
    print(fi.head(10))


[LightGBM] [Info] Number of positive: 23722, number of negative: 180555
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001526 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1319
[LightGBM] [Info] Number of data points in the train set: 204277, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.116127 -> initscore=-2.029633
[LightGBM] [Info] Start training from score -2.029633


d:\projects\home_default\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\projects\home_default\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/02/06 11:48:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
d:\projects\home_default\.venv\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


LightGBM Train ROC-AUC: 0.8135746503690642
LightGBM Test ROC-AUC: 0.757114318894974
LightGBM Test KS: 0.38131706020612055
                         feature  importance
1         num_scaled__LoanAmount        1898
0             num_scaled__Income        1864
3       num_scaled__InterestRate        1807
6   num_unscaled__MonthsEmployed        1701
2        num_scaled__CreditScore        1658
5              num_unscaled__Age        1440
4           num_scaled__DTIRatio        1191
12            ordinal__Education         445
7   num_unscaled__NumCreditLines         433
8         num_unscaled__LoanTerm         376


In [ ]:
with mlflow.start_run(run_name="Pipeline_XGBoost"):

    pipe_xgb = Pipeline(
        steps=[
            ("preprocess", preprocessor_tree),
            ("model", xgb.XGBClassifier(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="auc",
                random_state=42
            ))
        ]
    )

    pipe_xgb.fit(X_train, y_train)

    y_train_proba = pipe_xgb.predict_proba(X_train)[:, 1]
    y_test_proba = pipe_xgb.predict_proba(X_test)[:, 1]

    train_auc = roc_auc_score(y_train, y_train_proba)
    test_auc = roc_auc_score(y_test, y_test_proba)

    # ---- MLflow logging ----
    mlflow.log_params(pipe_xgb.named_steps["model"].get_params())
    mlflow.log_metric("train_roc_auc", train_auc)
    mlflow.log_metric("test_roc_auc", test_auc)

    # Additional metrics
    y_test_pred = (y_test_proba >= 0.5).astype(int)
    mlflow.log_metric("test_precision", precision_score(y_test, y_test_pred))
    mlflow.log_metric("test_recall", recall_score(y_test, y_test_pred))
    mlflow.log_metric("test_f1", f1_score(y_test, y_test_pred))
    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_test_pred))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_test_pred)
    cm_df = pd.DataFrame(cm, index=["Actual_0", "Actual_1"], columns=["Pred_0", "Pred_1"])
    mlflow.log_dict(cm_df.to_dict(), "confusion_matrix_xgb.json")

    # KS statistic and ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
    ks = np.max(tpr - fpr)
    mlflow.log_metric("test_ks", float(ks))
    roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr, "thresholds": thresholds})
    roc_df.to_csv("roc_curve_xgb.csv", index=False)
    mlflow.log_dict(roc_df.to_dict(), "roc_curve_xgb.json")

    mlflow.sklearn.log_model(pipe_xgb, artifact_path="model")

    # ---- Feature importance ----
    feature_names = pipe_xgb.named_steps["preprocess"].get_feature_names_out()

    fi = pd.DataFrame({
        "feature": feature_names,
        "importance": pipe_xgb.named_steps["model"].feature_importances_
    }).sort_values("importance", ascending=False)

    fi.to_csv("xgb_feature_importance.csv", index=False)
    mlflow.log_dict(fi.to_dict(), "xgb_feature_importance.json")

    print("XGBoost Train ROC-AUC:", train_auc)
    print("XGBoost Test ROC-AUC:", test_auc)
    print("XGBoost Test KS:", ks)
    print(fi.head(10))


2026/02/06 11:48:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
d:\projects\home_default\.venv\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


XGBoost Train ROC-AUC: 0.8290286738980631
XGBoost Test ROC-AUC: 0.7547363366546204
XGBoost Test KS: 0.379855559893306
                               feature  importance
5                    num_unscaled__Age    0.118569
3             num_scaled__InterestRate    0.067671
0                   num_scaled__Income    0.062192
13   nominal__EmploymentType_Full-time    0.055999
11                 binary__HasCoSigner    0.053024
10               binary__HasDependents    0.050982
6         num_unscaled__MonthsEmployed    0.050942
1               num_scaled__LoanAmount    0.050495
16  nominal__EmploymentType_Unemployed    0.045168
18      nominal__MaritalStatus_Married    0.038698


# Portfolio model without leaky features 

### train test split

In [55]:
# Dropping the leaky columns from the df

df_port = df.drop(['DTIRatio','InterestRate'],axis = 1)
X = df_port.drop('Default',axis = 1)
y = df_port['Default']


X_train,X_test,y_train,y_test= train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify=y,
    shuffle = True
)

### classification of features

In [56]:
num_unscaled_port = [
    "Age",
    "MonthsEmployed",
    "NumCreditLines",
    "LoanTerm"
]

num_scaled_port = [
    "Income",
    "LoanAmount",
    "CreditScore",
    # "InterestRate",
    # "DTIRatio"
]

ordinal_cols_port = [
    "Education"
]
nominal_cols_port = [
    "EmploymentType",
    "MaritalStatus",
    "LoanPurpose"
]
binary_cols_port = [
    "HasMortgage",
    "HasDependents",
    "HasCoSigner"
]

### preprocessor definition 

In [57]:

preprocessor_port = ColumnTransformer(
    transformers=[
        ("num_scaled", StandardScaler(), num_scaled_port),
        ("binary",binary_encoder, binary_cols_port),
        ("num_unscaled", "passthrough", num_unscaled_port),
        ("ordinal", ordinal_encoder, ordinal_cols_port),
        ("nominal", OneHotEncoder(handle_unknown="ignore", sparse_output=False,drop='first'), nominal_cols_port),
    ],
    remainder="drop",
    # verbose=True
)

preprocessor_tree_port = ColumnTransformer(
    transformers=[
        ("num_scaled", "passthrough", num_scaled_port),
        ("num_unscaled", "passthrough", num_unscaled_port),
        ("binary",binary_encoder, binary_cols_port),
        ("ordinal", ordinal_encoder, ordinal_cols_port),
        ("nominal", OneHotEncoder(handle_unknown="ignore", sparse_output=False), nominal_cols_port),
    ],
    remainder="drop",
    # verbose=True
)

### Baseline model

In [ ]:

import mlflow
import mlflow.sklearn

from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix
)
import pandas as pd
import numpy as np

with mlflow.start_run(run_name="Baseline_port"):
    pipe = Pipeline(
        steps=[
            ("preprocess", preprocessor_port),
            ("model", LogisticRegression(
                max_iter=1000,
                solver="lbfgs",
                n_jobs=-1
            ))
        ]
    )

    cv_scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=5,
        scoring="roc_auc"
    )
    pipe.fit(X_train,y_train)

    # Log model parameters
    mlflow.log_params(pipe.named_steps["model"].get_params())
    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("cv_scoring", "roc_auc")
    # mlflow.log_param("random_state", 42)
    mlflow.log_param("preprocessor", "StandardScaler + OrdinalEncoder + OneHot(drop=first) + BinaryOrdinal (no InterestRate/DTIRatio)")

    mlflow.log_metric("cv_roc_auc_mean", cv_scores.mean())
    mlflow.log_metric("cv_roc_auc_std", cv_scores.std())

    y_test_proba = pipe.predict_proba(X_test)[:, 1]

    test_roc_auc = roc_auc_score(y_test, y_test_proba)

    mlflow.log_metric("test_roc_auc", test_roc_auc)

    # Additional metrics
    y_test_pred = (y_test_proba >= 0.5).astype(int)
    mlflow.log_metric("test_precision", precision_score(y_test, y_test_pred))
    mlflow.log_metric("test_recall", recall_score(y_test, y_test_pred))
    mlflow.log_metric("test_f1", f1_score(y_test, y_test_pred))
    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_test_pred))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_test_pred)
    cm_df = pd.DataFrame(cm, index=["Actual_0", "Actual_1"], columns=["Pred_0", "Pred_1"])
    mlflow.log_dict(cm_df.to_dict(), "confusion_matrix_port.json")

    # KS statistic and ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
    ks = np.max(tpr - fpr)
    mlflow.log_metric("test_ks", float(ks))
    roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr, "thresholds": thresholds})
    roc_df.to_csv("roc_curve_port.csv", index=False)
    mlflow.log_dict(roc_df.to_dict(), "roc_curve_port.json")

    mlflow.sklearn.log_model(
        pipe,
        artifact_path="model"
    )

    print("CV ROC-AUC scores:", cv_scores)
    print("Mean ROC-AUC:", cv_scores.mean())
    print("test_roc_auc", test_roc_auc)
    print("test_ks", ks)


d:\projects\home_default\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\projects\home_default\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\projects\home_default\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\projects\home_default\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please lea

CV ROC-AUC scores: [0.72434685 0.71971418 0.71295044 0.71894158 0.71792141]
Mean ROC-AUC: 0.7187748919379295
test_roc_auc 0.7268869363147294
test_ks 0.33490518799105823


### logging in the LightBGM and XGboost 

In [ ]:
with mlflow.start_run(run_name="Pipeline_LightGBM_port"):

    pipe_lgb = Pipeline(
        steps=[
            ("preprocess",preprocessor_tree_port ),
            ("model", lgb.LGBMClassifier(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=-1,
                num_leaves=31,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary",
                random_state=42
            ))
        ]
    )

    pipe_lgb.fit(X_train, y_train)

    y_train_proba = pipe_lgb.predict_proba(X_train)[:, 1]
    y_test_proba = pipe_lgb.predict_proba(X_test)[:, 1]

    train_auc = roc_auc_score(y_train, y_train_proba)
    test_auc = roc_auc_score(y_test, y_test_proba)

    # ---- MLflow logging ----
    mlflow.log_params(pipe_lgb.named_steps["model"].get_params())
    mlflow.log_metric("train_roc_auc", train_auc)
    mlflow.log_metric("test_roc_auc", test_auc)

    # Additional metrics
    y_test_pred = (y_test_proba >= 0.5).astype(int)
    mlflow.log_metric("test_precision", precision_score(y_test, y_test_pred))
    mlflow.log_metric("test_recall", recall_score(y_test, y_test_pred))
    mlflow.log_metric("test_f1", f1_score(y_test, y_test_pred))
    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_test_pred))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_test_pred)
    cm_df = pd.DataFrame(cm, index=["Actual_0", "Actual_1"], columns=["Pred_0", "Pred_1"])
    mlflow.log_dict(cm_df.to_dict(), "confusion_matrix_lgb_port.json")

    # KS statistic and ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
    ks = np.max(tpr - fpr)
    mlflow.log_metric("test_ks", float(ks))
    roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr, "thresholds": thresholds})
    roc_df.to_csv("roc_curve_lgb_port.csv", index=False)
    mlflow.log_dict(roc_df.to_dict(), "roc_curve_lgb_port.json")

    mlflow.sklearn.log_model(pipe_lgb, artifact_path="model")

    # ---- Feature importance (post-preprocessing) ----
    feature_names = pipe_lgb.named_steps["preprocess"].get_feature_names_out()

    fi = pd.DataFrame({
        "feature": feature_names,
        "importance": pipe_lgb.named_steps["model"].feature_importances_
    }).sort_values("importance", ascending=False)

    fi.to_csv("lgb_feature_importance_port.csv", index=False)
    mlflow.log_dict(fi.to_dict(), "lgb_feature_importance_port.json")

    print("LightGBM Train ROC-AUC:", train_auc)
    print("LightGBM Test ROC-AUC:", test_auc)
    print("LightGBM Test KS:", ks)
    print(fi.head(10))


[LightGBM] [Info] Number of positive: 23722, number of negative: 180555
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001042 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 982
[LightGBM] [Info] Number of data points in the train set: 204277, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.116127 -> initscore=-2.029633
[LightGBM] [Info] Start training from score -2.029633


d:\projects\home_default\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\projects\home_default\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/02/06 11:49:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
d:\projects\home_default\.venv\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


LightGBM Train ROC-AUC: 0.787116399158875
LightGBM Test ROC-AUC: 0.7316792149350666
LightGBM Test KS: 0.34038951206559703
                         feature  importance
1         num_scaled__LoanAmount        2383
0             num_scaled__Income        2297
2        num_scaled__CreditScore        2090
4   num_unscaled__MonthsEmployed        2038
3              num_unscaled__Age        1746
10            ordinal__Education         579
6         num_unscaled__LoanTerm         552
5   num_unscaled__NumCreditLines         547
8          binary__HasDependents         331
9            binary__HasCoSigner         328


In [ ]:
with mlflow.start_run(run_name="Pipeline_XGBoost_port"):

    pipe_xgb = Pipeline(
        steps=[
            ("preprocess", preprocessor_tree_port),
            ("model", xgb.XGBClassifier(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="auc",
                random_state=42
            ))
        ]
    )

    pipe_xgb.fit(X_train, y_train)

    y_train_proba = pipe_xgb.predict_proba(X_train)[:, 1]
    y_test_proba = pipe_xgb.predict_proba(X_test)[:, 1]

    train_auc = roc_auc_score(y_train, y_train_proba)
    test_auc = roc_auc_score(y_test, y_test_proba)

    # ---- MLflow logging ----
    mlflow.log_params(pipe_xgb.named_steps["model"].get_params())
    mlflow.log_metric("train_roc_auc", train_auc)
    mlflow.log_metric("test_roc_auc", test_auc)

    # Additional metrics
    y_test_pred = (y_test_proba >= 0.5).astype(int)
    mlflow.log_metric("test_precision", precision_score(y_test, y_test_pred))
    mlflow.log_metric("test_recall", recall_score(y_test, y_test_pred))
    mlflow.log_metric("test_f1", f1_score(y_test, y_test_pred))
    mlflow.log_metric("test_accuracy", accuracy_score(y_test, y_test_pred))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_test_pred)
    cm_df = pd.DataFrame(cm, index=["Actual_0", "Actual_1"], columns=["Pred_0", "Pred_1"])
    mlflow.log_dict(cm_df.to_dict(), "confusion_matrix_xgb_port.json")

    # KS statistic and ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
    ks = np.max(tpr - fpr)
    mlflow.log_metric("test_ks", float(ks))
    roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr, "thresholds": thresholds})
    roc_df.to_csv("roc_curve_xgb_port.csv", index=False)
    mlflow.log_dict(roc_df.to_dict(), "roc_curve_xgb_port.json")

    mlflow.sklearn.log_model(pipe_xgb, artifact_path="model")

    # ---- Feature importance ----
    feature_names = pipe_xgb.named_steps["preprocess"].get_feature_names_out()

    fi = pd.DataFrame({
        "feature": feature_names,
        "importance": pipe_xgb.named_steps["model"].feature_importances_
    }).sort_values("importance", ascending=False)

    fi.to_csv("xgb_feature_importance_port.csv", index=False)
    mlflow.log_dict(fi.to_dict(), "xgb_feature_importance_port.json")

    print("XGBoost Train ROC-AUC:", train_auc)
    print("XGBoost Test ROC-AUC:", test_auc)
    print("XGBoost Test KS:", ks)
    print(fi.head(10))


2026/02/06 11:49:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
d:\projects\home_default\.venv\Lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


XGBoost Train ROC-AUC: 0.8043837704684134
XGBoost Test ROC-AUC: 0.7290718414069112
XGBoost Test KS: 0.3375463861120357
                               feature  importance
3                    num_unscaled__Age    0.122518
0                   num_scaled__Income    0.066950
11   nominal__EmploymentType_Full-time    0.060219
9                  binary__HasCoSigner    0.058423
1               num_scaled__LoanAmount    0.055467
4         num_unscaled__MonthsEmployed    0.054610
8                binary__HasDependents    0.054063
14  nominal__EmploymentType_Unemployed    0.050391
16      nominal__MaritalStatus_Married    0.043140
21           nominal__LoanPurpose_Home    0.040049


# Model Selection

LightGBM is selected as the final model because it outperforms XGBoost on key evaluation metrics such as:
- Test ROC-AUC
- KS statistic
- Precision score

The experiment results can be reviewed from the MLflow tracking data stored in the `mlruns` directory.

To inspect the runs and compare metrics, start the MLflow UI using the following command:

```bash
mlflow ui --backend-store-uri ./mlruns
